# 04.4 GPU and Efficiency

The goal of this notebook is not extreme performance tuning, but building a few of the most useful efficiency intuitions first.

Key concepts:

- `device
- `batch size
- `pin_memory
- `no_grad / eval`
- `automatic mixed precision

## Learning Goals

After this notebook, you should be able to:

1. Correctly place both the model and the data on the same `device`.
2. Understand how `batch size` affects the number of steps and training time.
3. Understand why `model.eval()` and `torch.no_grad()` are common for inference.
4. Know the typical use of `pin_memory
5. Build a first-layer understanding of automatic mixed precision.
6. Build a practical efficiency checklist.

In [ ]:
import time

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)
print("cuda available / cuda available =", torch.cuda.is_available())

## 1. Device Basics

The most basic and common source of mistakes is:

- the model and the data must be on the same device

In [ ]:
class TinyNet(nn.Module):
    def __init__(self, in_dim=128, hidden_dim=64, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)


model = TinyNet().to(device)
x_cpu = torch.randn(4, 128)
y_cpu = torch.tensor([0, 1, 0, 1], dtype=torch.long)

x = x_cpu.to(device)
y = y_cpu.to(device)
logits = model(x)

print("model device =", next(model.parameters()).device)
print("x device =", x.device)
print("y device =", y.device)
print("logits.shape =", logits.shape)

If you see an error like `Expected all tensors to be on the same device`, check this first.


## 2. Batch Size

`batch size` directly affects the number of steps per epoch.

In general:

- larger batches mean fewer steps
- but memory usage becomes higher
- bigger is not always better

In [ ]:
features = torch.randn(4096, 128)
labels = (features[:, 0] + 0.3 * features[:, 1] > 0).long()
dataset = TensorDataset(features, labels)


def sync_if_needed():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def benchmark_one_epoch(batch_size, device):
    torch.manual_seed(42)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=(device.type == "cuda"),
    )
    model = TinyNet().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
    loss_fn = nn.CrossEntropyLoss()

    sync_if_needed()
    start = time.perf_counter()
    total_items = 0
    total_loss = 0.0
    num_steps = 0

    model.train()
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=(device.type == "cuda"))
        yb = yb.to(device, non_blocking=(device.type == "cuda"))
        logits = model(xb)
        loss = loss_fn(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_items += xb.size(0)
        total_loss += loss.item() * xb.size(0)
        num_steps += 1

    sync_if_needed()
    elapsed = time.perf_counter() - start
    return {
        "batch_size": batch_size,
        "steps_per_epoch": num_steps,
        "elapsed_sec": round(elapsed, 4),
        "items_per_sec": round(total_items / elapsed, 2),
        "avg_loss": round(total_loss / total_items, 4),
    }


batch_results = [benchmark_one_epoch(bs, device) for bs in [16, 64, 256]]
batch_df = pd.DataFrame(batch_results)
batch_df

The two most important columns are:

- `steps_per_epoch`
- `elapsed_sec`

In real projects, you also need to consider memory limits, training stability, and throughput.


## 3. Inference Efficiency

Two common actions during inference are:

- `model.eval()`
- `torch.no_grad()`

`eval()` changes the behavior of layers like `Dropout / BatchNorm`; `no_grad()` avoids building the gradient graph.


In [ ]:
inference_model = TinyNet(hidden_dim=256).to(device)
inference_model.eval()
input_batch = torch.randn(2048, 128).to(device)
original_num_threads = torch.get_num_threads()
torch.set_num_threads(1)


def benchmark_inference(model, x, use_no_grad, repeats=120, warmup=20):
    context = torch.no_grad() if use_no_grad else torch.enable_grad()

    with context:
        for _ in range(warmup):
            out = model(x)

    sync_if_needed()
    start = time.perf_counter()
    with context:
        for _ in range(repeats):
            out = model(x)
    sync_if_needed()
    return time.perf_counter() - start, out.requires_grad


time_with_grad, grad_flag = benchmark_inference(inference_model, input_batch, use_no_grad=False)
time_no_grad, no_grad_flag = benchmark_inference(inference_model, input_batch, use_no_grad=True)
torch.set_num_threads(original_num_threads)

print("output.requires_grad with grad mode =", grad_flag)
print("output.requires_grad with no_grad =", no_grad_flag)
print("time_with_grad =", round(time_with_grad, 4), "sec")
print("time_no_grad =", round(time_no_grad, 4), "sec")
print("speedup ratio / speedup ratio =", round(time_with_grad / max(time_no_grad, 1e-8), 3))

`no_grad()` is not a magic switch that is always dramatically faster, but it explicitly disables gradient-graph construction.

On larger models, GPU inference, or longer batches, it usually uses less memory and is often more efficient.


## 4. DataLoader Settings

You do not need to tune `num_workers` aggressively from day one, but you should understand the intent of these parameters.


In [ ]:
def suggested_loader_kwargs(device):
    if device.type == "cuda":
        return {
            "num_workers": 2,
            "pin_memory": True,
            "note": "GPU training usually benefits from pinned memory and background workers.",
        }
    return {
        "num_workers": 0,
        "pin_memory": False,
        "note": "CPU or debugging mode usually starts with a simpler loader configuration.",
    }


loader_kwargs = suggested_loader_kwargs(device)
loader_kwargs

A practical memory rule:

- more useful for CPU -> GPU transfer
- often paired with pinned memory
- lets data preparation overlap more with training

## 5. Automatic Mixed Precision

The intuition of `AMP` is:

- let some computations run in lower precision
- to gain better throughput and lower memory usage

This is usually most useful during CUDA training.


In [ ]:
amp_loader = DataLoader(dataset, batch_size=128, shuffle=True, pin_memory=(device.type == "cuda"))
amp_model = TinyNet().to(device)
amp_loss_fn = nn.CrossEntropyLoss()
amp_optimizer = torch.optim.Adam(amp_model.parameters(), lr=0.01)

if device.type == "cuda":
    scaler = torch.cuda.amp.GradScaler()
    xb, yb = next(iter(amp_loader))
    xb = xb.to(device, non_blocking=True)
    yb = yb.to(device, non_blocking=True)

    amp_model.train()
    amp_optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits = amp_model(xb)
        loss = amp_loss_fn(logits, yb)

    scaler.scale(loss).backward()
    scaler.step(amp_optimizer)
    scaler.update()

    print("AMP step finished / AMP training step finished")
    print("loss =", float(loss))
else:
    print("CUDA not available / CUDA is not available in the current environment, so the real AMP demo is skipped.")
    print("But you should remember: AMP is common in GPU training, it improves throughput and reduces memory usage.")

In [ ]:
# Exercise 1
# Why does steps_per_epoch usually become smaller when batch size becomes larger?


Exercise 1 Reference Answer

Because each step processes more samples, the same dataset requires fewer steps.

But this does not mean larger batches are always better, because memory and optimization behavior also matter.


In [ ]:
# Exercise 2
# Why do we often use both model.eval() and torch.no_grad() during inference?


Exercise 2 Reference Answer

- BatchNorm` into inference mode
- avoids building the gradient graph and reduces overhead

They solve different problems, so they are often used together.


## Summary

The most important takeaways from this notebook are:

1. the model and the data must be on the same `device`
2. `batch size` affects step count, throughput, and memory usage
3. inference usually uses `model.eval()` + `torch.no_grad()`
4. non_blocking` are more about optimizing GPU data transfer
5. `AMP` is one common way to speed up GPU training